In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 09 · Inferencia — del modelo registrado a una función que cualquiera puede usar

El modelo registrado vive en MLflow; la app **no consulta MLflow ni Databricks**. Lo que hace falta es:

1. **Exportar** el modelo `champion` a un archivo (`models/model.joblib`) que viaja con el repositorio.
2. **Un esquema de entrada** (`models/input_schema.json`): qué datos se pueden preguntar y qué valores son *posibles*.
3. **Una predicción que valida**: nunca responde en silencio con un dato imposible.
4. **La ficha del modelo**: qué es, cuánto rinde (con intervalo), dónde falla y para qué **no** usarlo.

Basado en `material-del-curso/ejemplo-creditsight/notebooks/09_inferencia.ipynb`, con las cifras y variables de Eco-Score.

## 0. Datos y rutas

In [2]:
import json

import joblib
import mlflow
import numpy as np
import pandas as pd

from proyecto.config import (
    ANONYMOUS_GROUP_COL, CLASSES, COST_MATRIX, INPUT_LABELS, MODELS_DIR, PLAUSIBLE_RANGE, PROCESSED_DIR,
    PROSPER_RATING_START, REPORTS_DIR, TARGET,
)
from proyecto.inference.predict import (
    InputValidationError, fill_unused_columns, form_columns, input_frame, predict_with_scores, validate_inputs,
)
from proyecto.inference.schema import build_schema, save_schema
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    where = configure_mlflow()
MODEL_NAME = registered_model_name("eco_score_default")

train = pd.read_parquet(PROCESSED_DIR / "train_sin_memberkey.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_sin_memberkey.parquet")
numeric, categorical = form_columns()
results = json.loads((REPORTS_DIR / "07_metricas_test.json").read_text(encoding="utf-8"))
decisions = json.loads((REPORTS_DIR / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
registry = json.loads((REPORTS_DIR / "08_registro.json").read_text(encoding="utf-8"))
print("MLflow:", where, "· modelo:", MODEL_NAME)
print(f"entradas del formulario: {len(numeric)} numéricas y {len(categorical)} categóricas")

MLflow: local · modelo: eco_score_default
entradas del formulario: 33 numéricas y 6 categóricas


## 1. Exportar el modelo `champion`

Se pide al registro el modelo **por su alias** y se guarda en un archivo. Se exporta desde `champion` (no desde el
`final_pipeline.joblib` del notebook 07) para tener la certeza de que se entrega **lo que quedó registrado**.

In [3]:
model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
joblib.dump(model, MODELS_DIR / "model.joblib", compress=3)   # sin pérdida; GitHub rechaza archivos de más de 100 MB
size_mb = (MODELS_DIR / "model.joblib").stat().st_size / 1e6
print(f"models/model.joblib guardado ({size_mb:.1f} MB)")

models/model.joblib guardado (1.7 MB)


## 2. El esquema de entrada

La app y la predicción necesitan saber qué se puede preguntar: qué columnas, qué rango es **posible** y qué categorías
conoce el modelo. Dos cosas distintas:

- **Rango observado:** lo que se vio en el entrenamiento. Fuera de él, solo se **avisa**.
- **Rango posible** (`PLAUSIBLE_RANGE` en `src/proyecto/config.py`): lo que puede ocurrir en el mundo real. Es
  **conocimiento del analista**, no de los datos. Fuera de él, la predicción **se niega**.

**Qué se le pregunta a la persona.** El registro pide 43 columnas, pero 4 (`LoanOriginationDate`,
`FirstRecordedCreditLine`, `TotalProsperPaymentsBilled` y `ProsperPaymentsOneMonthPlusLate`) solo las lee `add_features`
para variables que el modelo final no usa. Por eso el formulario pregunta **39** y esas 4 se rellenan vacías. La sección 4
comprueba que eso no cambia ninguna probabilidad.

Se agregan dos **perfiles de ejemplo**, tomados de préstamos reales del entrenamiento (sin identificadores), para que
quien abra la app pueda empezar desde algo razonable.

In [4]:
inputs = input_frame(train)[numeric + categorical]


def profile(outcome: int) -> dict:
    """Un préstamo real de entrenamiento desde jul. 2009, con ese resultado y sin vacíos numéricos."""
    recent = pd.to_datetime(train["LoanOriginationDate"]) >= PROSPER_RATING_START
    pool = inputs[(train[TARGET] == outcome) & recent].dropna(subset=numeric)
    row = pool.iloc[len(pool) // 2]   # una fila de en medio, no un extremo
    return {c: (float(row[c]) if c in numeric else row[c]) for c in numeric + categorical}


examples = {"Préstamo real que se pagó": profile(0), "Préstamo real que cayó en impago": profile(1)}
schema = build_schema(
    inputs, numeric, categorical, CLASSES, cost_matrix=COST_MATRIX, valid_range=PLAUSIBLE_RANGE,
    labels=INPUT_LABELS, examples=examples,
)
schema["app_title"] = "Eco-Score · Riesgo de impago"
schema["responsible_use"] = (
    "- Este puntaje **prioriza la revisión** de una solicitud; la decisión final es de una persona. No rechaces de forma automática.\n"
    "- Se entrenó con préstamos de Prosper de 2005 a 2014 que ya terminaron: no lo uses con solicitudes actuales sin validarlo de nuevo.\n"
    "- Usa ingreso, vivienda y situación laboral: antes de un uso real debe revisarlo el área legal.\n"
    "- No usa el estado de residencia. El índice climático **no** entra al modelo.\n"
    "- Límites completos en `models/ficha_modelo.md`."
)
save_schema(schema, MODELS_DIR / "input_schema.json")

outside = [c for c in numeric if not inputs[c].dropna().between(*PLAUSIBLE_RANGE[c]).all()]
assert not outside, f"Hay datos reales fuera del rango posible: {outside}"
print(f"esquema guardado: {len(numeric)} numéricas, {len(categorical)} categóricas, clases {CLASSES}")
print("rangos posibles definidos para", len(schema["valid_range"]), "columnas; ningún dato de entrenamiento queda fuera")

esquema guardado: 33 numéricas, 6 categóricas, clases ['pagó', 'impago']
rangos posibles definidos para 33 columnas; ningún dato de entrenamiento queda fuera


**Lo que la app necesita además.** Los grupos del formulario y los nombres de las categorías salen de `src/proyecto/config.py`.
Las cifras de rendimiento se **copian** de `reports/07_metricas_test.json`, sin escribirlas a mano ni calcularlas de nuevo:
viajan dentro del esquema para que la app funcione solo con sus archivos, sin depender de la carpeta `reports/`.

In [5]:
from proyecto.config import FIELD_GROUPS, LISTING_CATEGORY_LABELS, YES_NO_LABELS

global_pr = next(r for r in results["intervalos"] if r["medida"] == "AUC-PR (mayor es mejor)")
recent_pr = results["desde_julio_2009"]
schema["field_groups"] = FIELD_GROUPS
schema["category_labels"] = {
    "ListingCategory (numeric)": LISTING_CATEGORY_LABELS,
    "IsBorrowerHomeowner": YES_NO_LABELS,
    "IncomeVerifiable": YES_NO_LABELS,
}
schema["performance"] = {
    "pr_auc": global_pr["valor"],
    "pr_auc_low": global_pr["IC 95 % inferior"],
    "pr_auc_high": global_pr["IC 95 % superior"],
    "pr_auc_recent": recent_pr["pr_auc"],
    "pr_auc_recent_low": recent_pr["ic95"][0],
    "pr_auc_recent_high": recent_pr["ic95"][1],
    "baseline_pr_auc": results["metricas_linea_base"]["pr_auc"],
    "flagged_rate": results["metricas_modelo"]["flagged_rate"],
    "recall_default": results["metricas_modelo"]["recall_default"],
    "precision_default": results["metricas_modelo"]["precision_default"],
}
save_schema(schema, MODELS_DIR / "input_schema.json")
print("esquema ampliado con:", ", ".join(k for k in schema if k in ("field_groups", "category_labels", "performance")))
print("rendimiento copiado de reports/07_metricas_test.json:", schema["performance"])

esquema ampliado con: field_groups, category_labels, performance
rendimiento copiado de reports/07_metricas_test.json: {'pr_auc': 0.6208753232643422, 'pr_auc_low': 0.6043624001027884, 'pr_auc_high': 0.6407050463423962, 'pr_auc_recent': 0.4788829281257714, 'pr_auc_recent_low': 0.4480196739895029, 'pr_auc_recent_high': 0.5102232168336226, 'baseline_pr_auc': 0.31036222379174655, 'flagged_rate': 0.6374735142770659, 'recall_default': 0.8751625487646294, 'precision_default': 0.42608420386198165}


**Qué se pide a la vista.** Del total de 39 datos, la app pide 12 esenciales y deja el resto como opcional. Salen del peso
SHAP de la etapa 7, sin `CreditGrade`, que solo existe antes de 2009, y con el monto de la solicitud.

Los opcionales que no se llenan se completan con valores típicos, y eso tiene un costo. Se mide aquí con los **préstamos
reservados** de la etapa 6, que no se usan para entrenar este modelo de medición. **La prueba no se usa.** La cifra viaja en el
esquema para que la app la muestre.

In [6]:
from proyecto.config import ESSENTIAL_FIELDS
from proyecto.data.split import split_by_member
from proyecto.features.engineering import FINAL_FEATURES
from proyecto.inference.predict import blank_unasked
from proyecto.models.train import build_pipeline, encode_target, metrics_from_proba
from proyecto.models.tuning import MODELS

tune_df, val_df = split_by_member(train, test_size=0.25, seed=7, group_col=ANONYMOUS_GROUP_COL)
assert len(val_df) == decisions["reserva"]["prestamos"], "La reserva no es la de la etapa 6"
assert not set(val_df[ANONYMOUS_GROUP_COL]) & set(test[ANONYMOUS_GROUP_COL])

_, make_fn, scale = MODELS[decisions["modelo_final"]]
reserve_model = build_pipeline(make_fn(decisions["parametros"]), scale=scale, **FINAL_FEATURES)
reserve_model.fit(input_frame(tune_df), encode_target(tune_df[TARGET]))

y_reserve = encode_target(val_df[TARGET])
is_recent = (pd.to_datetime(val_df["LoanOriginationDate"]) >= PROSPER_RATING_START).to_numpy()
all_fields = input_frame(val_df)[numeric + categorical]
proba_all = reserve_model.predict_proba(fill_unused_columns(all_fields))[:, 1]
proba_essential = reserve_model.predict_proba(fill_unused_columns(blank_unasked(all_fields, ESSENTIAL_FIELDS)))[:, 1]

schema["essential_fields"] = ESSENTIAL_FIELDS
schema["reduced_input"] = {
    "pr_auc_all": metrics_from_proba(y_reserve, proba_all)["pr_auc"],
    "pr_auc_essential": metrics_from_proba(y_reserve, proba_essential)["pr_auc"],
    "pr_auc_recent_all": metrics_from_proba(y_reserve[is_recent], proba_all[is_recent])["pr_auc"],
    "pr_auc_recent_essential": metrics_from_proba(y_reserve[is_recent], proba_essential[is_recent])["pr_auc"],
    "cost_all": metrics_from_proba(y_reserve, proba_all)["cost"],
    "cost_essential": metrics_from_proba(y_reserve, proba_essential)["cost"],
    "decisions_changed": float(((proba_all > 0.2) != (proba_essential > 0.2)).mean()),
}
save_schema(schema, MODELS_DIR / "input_schema.json")
assert abs(schema["reduced_input"]["pr_auc_all"] - decisions["reserva"]["pr_auc"]) < 1e-9, "No reproduce la reserva de la etapa 7"
print(f"datos esenciales: {len(ESSENTIAL_FIELDS)} de {len(numeric + categorical)}")
print(pd.Series(schema["reduced_input"]).round(4).to_string())

datos esenciales: 12 de 39
pr_auc_all                 0.6080
pr_auc_essential           0.5816
pr_auc_recent_all          0.4898
pr_auc_recent_essential    0.4382
cost_all                   0.5073
cost_essential             0.5424
decisions_changed          0.1142


## 3. Probar la predicción

Tres casos que importan:

- una fila **normal** → responde;
- una fila **incompleta** (la persona marcó "no sé") → también responde, porque el pipeline imputa;
- una fila con un dato **imposible** → **se niega y explica por qué**.

`decision` usa la regla de menor costo (marca `impago` si P(impago) > 0,2). No es un rechazo: significa **enviar a revisión**.
Se agrega un cuarto caso: un valor raro pero posible, que responde con una **advertencia**.

In [7]:
normal = pd.DataFrame([examples["Préstamo real que se pagó"]])
print("— fila normal —")
print(predict_with_scores(normal, model, schema).round(3).T.to_string(header=False))

incomplete = normal.copy()
incomplete.loc[:, "StatedMonthlyIncome"] = np.nan
print("\n— con un dato faltante (StatedMonthlyIncome) —")
print(predict_with_scores(incomplete, model, schema)[["decision", "proba_impago", "needs_review"]].round(3).to_string(index=False))

impossible = normal.copy()
impossible.loc[:, "CreditScoreRangeLower"] = -500
print("\n— con un dato imposible (CreditScoreRangeLower = -500) —")
try:
    predict_with_scores(impossible, model, schema)
    raise AssertionError("debía negarse a predecir con un puntaje de -500")
except InputValidationError as error:
    print("se niega, como debe:", error)

unusual = normal.copy()
unusual.loc[:, "AmountDelinquent"] = 900_000.0
print("\n— con un valor raro pero posible (AmountDelinquent = 900.000) —")
print(validate_inputs(unusual, schema))

— fila normal —
decision        pagó
proba_pagó     0.934
proba_impago   0.066
most_likely     pagó
expected_cost  0.263
needs_review   False

— con un dato faltante (StatedMonthlyIncome) —
decision  proba_impago  needs_review
    pagó         0.077         False

— con un dato imposible (CreditScoreRangeLower = -500) —
se niega, como debe: Puntaje de crédito (límite inferior): debe estar entre 300 y 900

— con un valor raro pero posible (AmountDelinquent = 900.000) —
['Monto en mora (USD): fuera del rango visto en el entrenamiento (0 a 444.745)']


## 4. Comprobar que el archivo exportado es el modelo que se evaluó

Antes de dárselo a la app: `models/model.joblib` debe dar **las mismas probabilidades** que se obtuvieron al abrir la
prueba en el notebook 07. Esto **no es una nueva lectura de la prueba para decidir algo**: no se calcula ninguna métrica, es
una comprobación de identidad contra lo que ese notebook guardó.

Se comprueba de la forma en que lo usará la app: solo con las 39 columnas del formulario y las otras 4 vacías.

In [8]:
exported = joblib.load(MODELS_DIR / "model.joblib")
saved_proba = joblib.load(PROCESSED_DIR / "prueba_probabilidades.joblib")["final"]
form_rows = input_frame(test)[numeric + categorical]
current_proba = exported.predict_proba(fill_unused_columns(form_rows))
identical = bool(np.allclose(saved_proba, current_proba))
print("probabilidades idénticas a las del notebook 07:", identical, f"(diferencia máxima {np.abs(saved_proba - current_proba).max():.2e})")
assert identical, "El model.joblib exportado no es el modelo que se evaluó"

probabilidades idénticas a las del notebook 07: True (diferencia máxima 0.00e+00)


## 5. La ficha del modelo

Una ficha dice **qué es**, **cuánto rinde (con su intervalo)** y **dónde no debe usarse**. Se genera con las cifras de
`reports/07_metricas_test.json` y `reports/08_registro.json`, no a mano, para que no pueda contradecir a esos reportes.

In [9]:
intervals = {r["medida"]: r for r in results["intervalos"]}
pr, cost = intervals["AUC-PR (mayor es mejor)"], intervals["costo por solicitud (menor es mejor)"]
d_pr, d_cost = intervals["AUC-PR: modelo − línea base"], intervals["costo: modelo − línea base"]
m, base = results["metricas_modelo"], results["metricas_linea_base"]
recent, climate = results["desde_julio_2009"], results["indice_climatico"]
top = pd.read_csv(REPORTS_DIR / "07_importancia_shap.csv", index_col=0).squeeze().head(5)


def es(value: float, digits: int = 4) -> str:
    """Número con coma decimal."""
    return f"{value:.{digits}f}".replace(".", ",")


def thousands(value: float) -> str:
    """Entero con punto de miles."""
    return f"{value:,.0f}".replace(",", ".")


top_text = ", ".join(f"`{name}` ({es(value)})" for name, value in top.items())

card = f"""# Ficha del modelo — Eco-Score

## Qué es

Estima la probabilidad de **impago** de un préstamo personal de Prosper a partir del perfil financiero del prestatario y de las
condiciones del préstamo. Apoya al analista de riesgo a **priorizar la revisión**: no decide por sí solo y la decisión final es
siempre humana.

- **Modelo:** {registry["modelo"]} con parámetros optimizados, {"calibrado" if registry["calibrado"] else "sin calibrar"}.
- **Registro:** `{registry["modelo_registrado"]}` versión {registry["version"]}, alias `{registry["alias"]}` (MLflow {registry["donde"]}). El archivo para la app es `models/model.joblib`.
- **Regla de decisión:** menor costo esperado con la matriz 4 a 1: marcar para revisión si P(impago) > 0,2.

## Datos de entrenamiento

- **Fuente y licencia:** *Prosper Loan Data* (Kaggle, Henry Okam), CC0: Public Domain.
- **Tamaño:** {thousands(len(train))} préstamos de {thousands(train[ANONYMOUS_GROUP_COL].nunique())} prestatarios (nov. 2005 a mar. 2014), solo préstamos terminados.
- **Partición:** por prestatario. La prueba ({thousands(results["prueba"]["prestamos"])} préstamos de {thousands(results["prueba"]["prestatarios"])} prestatarios) se abrió **una sola vez**, con las decisiones congeladas antes ({decisions["fecha_y_hora"]}).
- **Datos personales:** no hay nombres ni documentos. Las llaves del préstamo y `Occupation` se eliminaron; `MemberKey` se cambió por un número anónimo que no entra al modelo.

## Rendimiento (prueba, prestatarios nunca vistos)

| Métrica | Modelo | Línea base | Intervalo del 95 % (modelo) |
|---|---|---|---|
| AUC-PR de impago (principal) | {es(pr["valor"])} | {es(base["pr_auc"])} | {es(pr["IC 95 % inferior"])} a {es(pr["IC 95 % superior"])} |
| Costo por solicitud, matriz 4 a 1 (menor es mejor) | {es(cost["valor"])} | {es(base["cost"])} | {es(cost["IC 95 % inferior"])} a {es(cost["IC 95 % superior"])} |
| AUC-PR desde julio de 2009 | {es(recent["pr_auc"])} | no calculada | {es(recent["ic95"][0])} a {es(recent["ic95"][1])} |

- **Frente a la línea base:** AUC-PR +{es(d_pr["valor"])} (IC 95 %: +{es(d_pr["IC 95 % inferior"])} a +{es(d_pr["IC 95 % superior"])}) y costo {es(d_cost["valor"])} (IC 95 %: {es(d_cost["IC 95 % inferior"])} a {es(d_cost["IC 95 % superior"])}), unos {thousands(results["ahorro_usd_por_1000_solicitudes"])} USD por cada 1.000 solicitudes. La conversión a dólares es una aproximación.
- **Carga de trabajo:** marca para revisión el {es(m["flagged_rate"] * 100, 1)} % de las solicitudes y detecta el {es(m["recall_default"] * 100, 1)} % de los impagos. De cada 100 marcadas, {m["precision_default"] * 100:.0f} caen en impago, y {results["carga"]["impagos_sin_revision"]} impagos pasan sin revisión.

## Variables que más pesan

{top_text} (|SHAP| medio). `BorrowerAPR` y `EstimatedEffectiveYield` las calcula Prosper: el modelo depende en buena parte del riesgo
que la plataforma ya había puesto en el precio. SHAP explica al modelo, no al mundo: una importancia alta no es una causa.

## Límites y usos no recomendados

- **Datos de una época:** préstamos de Prosper de 2005 a 2014, solo terminados. Los recientes que ya terminaron son sobre todo los que terminaron rápido.
- **Espera el rendimiento de {es(recent["pr_auc"])}, no el de {es(pr["valor"])}**, en solicitudes parecidas a las actuales: la cifra global se apoya en préstamos antiguos con más impagos.
- **Supuesto de negocio:** el modelo usa el puntaje y el precio de Prosper, disponibles al crear la solicitud. Si aún no existen, los campos quedan en "no sé" y el modelo imputa.
- **Índice climático:** no entra al modelo. Con él, el AUC-PR de la prueba sube {es(climate["diferencia_pr_auc"])} (IC 95 %: {es(climate["ic95_pr_auc"][0])} a {es(climate["ic95_pr_auc"][1])}) y el costo no se distingue del ruido. Es un supuesto académico del grupo, sin fuente verificable, y no muestra que el clima cause impagos.
- **Atributos sensibles o de uso regulado:** ingreso declarado, rango de ingreso, situación laboral y vivienda propia. Deben revisarse con el área legal, junto con un análisis de trato desigual, antes de cualquier uso real. `BorrowerState` no entra al modelo.
- **No debe usarse para:** rechazar solicitudes de forma automática, penalizar a alguien por su estado de residencia ni tomar decisiones reales sin validarlo con datos posteriores a 2014.
- **No se evaluó:** estabilidad en el tiempo, deriva y préstamos que aún no habían terminado.

## Reproducibilidad

Código en `src/`, pruebas en `tests/`, notebooks en `notebooks/`. Esta ficha se genera con `notebooks/09_inferencia.ipynb`.
"""
(MODELS_DIR / "ficha_modelo.md").write_text(card, encoding="utf-8")
print(card)

# Ficha del modelo — Eco-Score

## Qué es

Estima la probabilidad de **impago** de un préstamo personal de Prosper a partir del perfil financiero del prestatario y de las
condiciones del préstamo. Apoya al analista de riesgo a **priorizar la revisión**: no decide por sí solo y la decisión final es
siempre humana.

- **Modelo:** lightgbm con parámetros optimizados, sin calibrar.
- **Registro:** `eco_score_default` versión 1, alias `champion` (MLflow local). El archivo para la app es `models/model.joblib`.
- **Regla de decisión:** menor costo esperado con la matriz 4 a 1: marcar para revisión si P(impago) > 0,2.

## Datos de entrenamiento

- **Fuente y licencia:** *Prosper Loan Data* (Kaggle, Henry Okam), CC0: Public Domain.
- **Tamaño:** 39.641 préstamos de 32.341 prestatarios (nov. 2005 a mar. 2014), solo préstamos terminados.
- **Partición:** por prestatario. La prueba (9.911 préstamos de 8.111 prestatarios) se abrió **una sola vez**, con las decisiones congeladas antes (2026-11-05T20

## 6. Siguiente: la app

La app carga `models/model.joblib` y `models/input_schema.json` y no consulta Databricks. Desde la raíz del repositorio:

```bash
streamlit run app/streamlit_app.py
```

Dejar un campo vacío equivale a *No sé*, y un valor imposible muestra un error claro. La app pide 12 datos esenciales y deja el resto plegado como opcional.